# 01 · Load raw data into SQL

Loads all 9 raw Olist CSVs into a SQLite database (`db/olist.db`) using
the DDL in `sql/schema_sqlite.sql`, so joins/aggregations happen in SQL
first rather than with a chain of pandas `.merge()` calls.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
DB_DIR = ROOT / "db"
DB_DIR.mkdir(exist_ok=True)
DB_PATH = DB_DIR / "olist.db"
SCHEMA_PATH = ROOT / "sql" / "schema_sqlite.sql"

TABLE_FILE_MAP = {
    "category_translation": "product_category_name_translation.csv",
    "customers":            "olist_customers_dataset.csv",
    "geolocation":          "olist_geolocation_dataset.csv",
    "sellers":              "olist_sellers_dataset.csv",
    "products":             "olist_products_dataset.csv",
    "orders":               "olist_orders_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "order_payments":       "olist_order_payments_dataset.csv",
    "order_reviews":        "olist_order_reviews_dataset.csv",
}

# Load order matters because of the FK dependencies declared in the schema.
LOAD_ORDER = [
    "category_translation", "customers", "geolocation", "sellers",
    "products", "orders", "order_items", "order_payments", "order_reviews",
]

In [2]:
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
conn.executescript(SCHEMA_PATH.read_text())

# FK enforcement is OFF for this raw landing load on purpose, so we can
# detect orphaned rows (e.g. product categories with no translation) with
# an explicit integrity check afterward, instead of a silent reject.
conn.execute("PRAGMA foreign_keys = OFF")

print(f"Loading into {DB_PATH} ...")
total_rows = 0
for table in LOAD_ORDER:
    csv_path = RAW / TABLE_FILE_MAP[table]
    df = pd.read_csv(csv_path)
    df.to_sql(table, conn, if_exists="append", index=False)
    total_rows += len(df)
    print(f"  {table:<22} {len(df):>8,} rows  <- {csv_path.name}")

conn.commit()
print(f"\nTotal source rows loaded: {total_rows:,}")

Loading into /home/claude/project_final/ecommerce-analytics-project/db/olist.db ...
  category_translation         71 rows  <- product_category_name_translation.csv


  customers                99,441 rows  <- olist_customers_dataset.csv


  geolocation            1,000,163 rows  <- olist_geolocation_dataset.csv
  sellers                   3,095 rows  <- olist_sellers_dataset.csv
  products                 32,951 rows  <- olist_products_dataset.csv


  orders                   99,441 rows  <- olist_orders_dataset.csv


  order_items             112,650 rows  <- olist_order_items_dataset.csv


  order_payments          103,886 rows  <- olist_order_payments_dataset.csv


  order_reviews           100,000 rows  <- olist_order_reviews_dataset.csv

Total source rows loaded: 1,551,698


## Sanity checks
Row counts per table, a real join across 4 tables, and a foreign-key
integrity check on product categories.

In [3]:
cur = conn.cursor()
print("Row counts in DB:")
for table in LOAD_ORDER:
    n = cur.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<22} {n:>8,}")

Row counts in DB:
  category_translation         71
  customers                99,441
  geolocation            1,000,163
  sellers                   3,095
  products                 32,951
  orders                   99,441
  order_items             112,650
  order_payments          103,886
  order_reviews           100,000


In [4]:
print("Sanity join (orders -> customers -> order_items -> products), 3 rows:")
sample = cur.execute("""
    SELECT o.order_id, c.customer_state, oi.price, p.product_category_name
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p ON p.product_id = oi.product_id
    LIMIT 3
""").fetchall()
for row in sample:
    print(" ", row)

Sanity join (orders -> customers -> order_items -> products), 3 rows:
  ('00010242fe8c5a6d1ba2dd792cb16214', 'RJ', 58.9, 'cool_stuff')
  ('00018f77f2f0320c557190d7a144bdd3', 'SP', 239.9, 'pet_shop')
  ('000229ec398224ef6ca0657da4fc703e', 'MG', 199.0, 'moveis_decoracao')


In [5]:
print("Product categories with no English translation on file:")
orphans = cur.execute("""
    SELECT DISTINCT p.product_category_name, COUNT(*) AS n_products
    FROM products p
    LEFT JOIN category_translation t
           ON t.product_category_name = p.product_category_name
    WHERE p.product_category_name IS NOT NULL
      AND t.product_category_name IS NULL
    GROUP BY p.product_category_name
""").fetchall()
for cat, n in orphans:
    print(f"  '{cat}' -> {n} products")
if not orphans:
    print("  none found")

conn.close()

Product categories with no English translation on file:
  'pc_gamer' -> 3 products
  'portateis_cozinha_e_preparadores_de_alimentos' -> 10 products
